# Q1d — Mean+Max pool word2vec → two-layer MLP

**W&B run:** _filled in after first run_

The "extra experiment" required by the spec. Same network as Q1b, same 20-epoch word2vec model — only the pooling changes from mean to mean⊕max (concatenation), doubling the input dimension from 100 to 200.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb
from gensim.models import KeyedVectors
from torch.utils.data import DataLoader, TensorDataset

from nlp_project import SEED, set_seed
from nlp_project.data import load_20ng, preprocess, train_val_split
from nlp_project.embeddings import mean_max_pool
from nlp_project.eval import evaluate, plot_confusion
from nlp_project.model import MLP
from nlp_project.train import train as train_loop

set_seed()
FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path("../models")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class _W2VWrapper:
    """mean_max_pool only needs `.wv` and `.vector_size`."""
    def __init__(self, wv: KeyedVectors) -> None:
        self.wv = wv
        self.vector_size = wv.vector_size

w2v = _W2VWrapper(KeyedVectors.load(str(MODEL_DIR / "w2v_epoch20.kv")))


In [ ]:
# Preprocess + vectorize (mean+max pool of w2v vectors → 200-dim doc vectors).
train_docs, train_labels, test_docs, test_labels, label_names = load_20ng(remove=True)
train_tokens = preprocess(train_docs, drop_stopwords=False)
test_tokens = preprocess(test_docs, drop_stopwords=False)

X_train_full = mean_max_pool(train_tokens, w2v)
X_test = mean_max_pool(test_tokens, w2v)
print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")


In [ ]:
# Split, loaders, train.
X_train, y_train, X_val, y_val = train_val_split(
    list(X_train_full), train_labels, val_frac=0.1, seed=SEED,
)
X_train, X_val = np.asarray(X_train), np.asarray(X_val)

def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, 64, shuffle=True)
val_loader = make_loader(X_val, y_val, 64, shuffle=False)
test_loader = make_loader(X_test, test_labels, 64, shuffle=False)

run = wandb.init(
    project="hslu-nalapro-q1",
    name="q1d-word2vec-mean-max-pool",
    config={"vectorizer": "word2vec-mean+max-pool", "vector_size": 100,
            "in_dim": 200, "hidden_dim": 256, "dropout": 0.3, "lr": 1e-3,
            "batch_size": 64},
)
model = MLP(in_dim=200, hidden_dim=256, num_classes=20, dropout=0.3)
history = train_loop(
    model, train_loader, val_loader,
    epochs=50, lr=1e-3, device=DEVICE, wandb_run=run, patience=5,
)


In [ ]:
# Evaluate, plot, finish run.
metrics = evaluate(model, test_loader, label_names, device=DEVICE)
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test macro-F1: {metrics['macro_f1']:.4f}")
plot_confusion(
    metrics["confusion_matrix"], label_names,
    save_path=FIG_DIR / "confusion_matrix_q1d.png",
    title="Q1d — word2vec mean+max pool",
)
run.log({"test_accuracy": metrics["accuracy"], "test_macro_f1": metrics["macro_f1"]})
run.finish()


In [ ]:
# Comparison table CSV.
# Replace the placeholder values below with the numbers printed by Q1b and Q1c
# before running this cell, then re-run.
Q1B_ACC, Q1B_F1 = None, None  # paste from q1b_word2vec.ipynb output
Q1C_ACC, Q1C_F1 = None, None  # paste from q1c_tfidf.ipynb output

rows = [
    {"experiment": "Q1b — word2vec mean-pool",     "accuracy": Q1B_ACC, "macro_f1": Q1B_F1},
    {"experiment": "Q1c — TF-IDF",                 "accuracy": Q1C_ACC, "macro_f1": Q1C_F1},
    {"experiment": "Q1d — word2vec mean+max-pool", "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]},
]
df = pd.DataFrame(rows)
df.to_csv(FIG_DIR / "metric_comparison_table.csv", index=False)
df


## Notes for the report

- The mean+max-pool variant doubles the input dim with zero changes to the network's parameter count beyond the first linear layer. Discuss whether the bump in macro-F1 (if any) justifies the extra dimensions.
- Comparison table goes straight into the report.
